## Module 2: FSHD Clinical Trial Project 

Complete the sections below as a team. Remove the word “Template” only if it appears in the final notebook title.

## Project Goal:

The goal of this project is to determine which methods are most effective for tracking disease progression and measuring treatment effects in patients with facioscapulohumeral muscular dystrophy (FSHD). Because FSHD progresses slowly and can affect each side of the body differently, it can be difficult to detect meaningful changes during a short clinical trial. We will compare traditional clinical and functional assessments, including the clinical severity score, Timed Up and Go (TUG), 10-meter walk/run, 4-step stair ascent and descent, and dorsiflexion strength measured by quantitative muscle testing (QMT). We will also explore newer methods, including quantitative MRI measurements of muscle fat fraction, fat-fraction variability, and edema using quantitative STIR, as well as the baseline DUX4 gene signature. By comparing these methods, we hope to identify which measures may be most sensitive to disease progression and should be prioritized as exploratory endpoints in future FSHD clinical trials.

## Disease Background: 
- ***Please provide citations and/or hyperlinks to where you obtained information (including AI) for each of the bullet points.***

*The following 7 bullets seem most relevant, but feel free to choose others if you like:*

Prevalence & incidence- Most prevalent hereditary muscular dystrophy affecting men, women and children. A conservative estimate of incidence for the most common type of FSHD is 1 in 14,286 births throughout the world. However, due to increased experience with FSHD, population based research and improved genetic testing, this estimate may be low. Actual incidence may be 1 in 7,500.

Symptoms- FSHD causes progressive loss, wasting and atrophy of all skeletal muscles. Muscle atrophy and functional loss occur asymmetrically, with one side being more affected than the other. FSHD displays a characteristic gradual spread of muscle involvement, involving the face, shoulder, girdle, upper-arm musculature and the abdominal and foot-extensor muscles. The most common initial symptom is difficulty reaching above shoulder level. Less common symptoms are foot drop, asymptomatic scapular fixation, and facial weakness. Truncal weakness is an early and frequent manifestation that is easily overlooked during examination. Weak abdominal muscles result in protuberant abdomen and contribute to lumbar lordosis. Hearing loss (high frequency bilateral sensorineural), vision problems and seizures. 

Diagnosis- prenatal genetic testing is available for FSHD. Pre-implantation genetic diagnosis is available for couples using in vitro fertilization.  



## Data Set:

Describe the FSHD dataset used in your project. Include:

- The subject and visit/timepoint(s) analyzed
- The tibialis anterior MRI data
- The water-image and fat-image files
- The tibialis anterior segmentation mask
- The sampled locations along the muscle
The sampled muscle regions are: dorsiflexion right and left (ff_baseline_R and ff_baseline_L). The dataset does not specify the exact muscle name. 

The physical distance from the muscle start is coded in distance_from_muscle_start_mm.

- The meaning and units of the image signals
Ff: fat fraction and the proportion of muscle tissue occupied by fat in percentage (fat/ (fat+water)x100) per pixel (Wong et al., 2024; Ma, 2008).

Fvar: fat-fraction variability, how fat fraction varies within the sampled muscle region. 

Stir_quant: quantitative STIR signal, an MRI-derived measure that helps evaluate muscle edema or tissue water content increment. 

Mask a uint8array with 1 for TA pixels. 

- The clinical, biomarker, or functional measurements included in the dataset
The dataset contains genetic information, MRI biomarkers, muscle strength testing, and functional assessments. All measurements are recorded at baseline, Year 1 and Year 2. 

For genetic and molecular biomarkers- d4z4, dux4_signature_baseline_R and dux4_signature_baseline_L are variables that relate to the genetic and/or molecular characteristics of the participants. 

The variables for MRI-derived biomarkers are ff_baseline_R, ff_y1_R, ff_baseline_:, ff_y1_L, ff_y2_L, fvar, and stir_quant. These variables characterize muscle composition and changes over time. 

The variables for muscle strength and physical function are dorsiflexion_qmt, tug_time, wr_10, ascend_4, descend_4, and clinical_severity_score. These variables are the information about how muscle disease affects their everyday movement and mobility. 

The variables for demographic and visit-timing variables are subject_id, sex, age_baseline, lag_months_baseline, lag_months_y1, and lag_months_y2. 



- The source of the data

Regional and bilateral MRI and gene signatures in facioscapulohumeral dystrophy: implications for clinical trial design and mechanisms of disease progression. Human Molecular Genetics, 33(8), 698–708. https://doi.org/10.1093/hmg/ddae007[cite: 20] 

Explain how the MRI data and muscle mask were generated or provided for this project. Cite the paper, dataset documentation, or other sources that informed your description.

## MRI Data Analysis Methdology:

Describe the methods used to analyze the FSHD data. Include the Python code, tables, and figures needed for another researcher to reproduce your analysis.  Use the example from class to create this section.  NOte that you will not anaylze all the data that goes into the final project analysis.  Rather you are reproducing how one dataset was generated and using that to describe the methodology.

For the MRI analysis, explain:

- How the water, fat, and segmentation-mask files were loaded
- How the tibialis anterior mask was used to identify the muscle
- How fat fraction was calculated for the whole image region and for the TA muscle only


In [ ]:
#How the fat-fraction distribution was summarized
#The fat-fraction distribution is showcased in a histogram. The graph independent value is the fat fraction in percentage and the y value is the number of pixels. 
#The code uses whole_image_values and muscle_values, which are created in an earlier code cell. 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("FSHD_TA_MRI_data_0020_TP1_TP2")
manifest = pd.read_csv(DATA_DIR / "slice_info.csv")

row = manifest[manifest["timepoint"] == "TP1"].iloc[3]  # representative mid-muscle slice

water = np.load(DATA_DIR / row["water_file"])
fat = np.load(DATA_DIR / row["fat_file"])
mask = np.load(DATA_DIR / row["mask_file"]).astype(bool)
denominator = water + fat
valid_pixels = denominator > 0

fat_fraction = np.divide(
    fat,
    denominator,
    out=np.zeros_like(fat, dtype=float),
    where=valid_pixels
) * 100

# Whole supplied image region
fat_fraction_full = fat_fraction.copy()

# TA muscle only (pixels outside the mask set to NaN)
fat_fraction_masked = fat_fraction.copy()
fat_fraction_masked[~mask] = np.nan

whole_image_values = fat_fraction_full[valid_pixels]
muscle_values = fat_fraction_masked[~np.isnan(fat_fraction_masked)]
plt.figure(figsize=(7, 4))
plt.hist(whole_image_values, bins=30, alpha=0.55, label="Whole supplied region")
plt.hist(muscle_values, bins=30, alpha=0.65, label="TA muscle only")
plt.xlabel("Fat fraction (%)")
plt.ylabel("Number of pixels")
plt.title("Fat-fraction distributions")
plt.legend()
plt.show()


In [ ]:
# Which summary statistics were calculated, such as mean, median, standard deviation, and number of muscle pixels

# The percent along muscle, distance from muscle start the mean fat fraction percent, the median fat fraction percent and the standard deviation fraction percent were calculated by the results.append. Additionally, the distance from muscle start, mean TA fat fraction (%), mean fat fraction along the tibialis anterior. 

# Only four things are calculated, the mean, median, standard deviation, and number of muscle pixels. 

# The pixel count shows how much muscle is in each slice. Each row of the results table also records the subject, visit, slice ID, percent along the muscle, and distance from the muscle (mm). 
def calculate_fat_fraction(water, fat, mask):
    """Return a masked fat-fraction image in percent."""
    denominator = water + fat
    result = np.divide(
        fat,
        denominator,
        out=np.zeros_like(fat, dtype=float),
        where=denominator > 0
    ) * 100
    result[~mask] = np.nan
    return result

analysis_timepoint = "TP1"  # analyze one visit at a time
analysis_manifest = manifest[manifest["timepoint"] == analysis_timepoint]
results = []

for _, row in analysis_manifest.iterrows():
    water = np.load(DATA_DIR / row["water_file"])
    fat = np.load(DATA_DIR / row["fat_file"])
    mask = np.load(DATA_DIR / row["mask_file"]).astype(bool)
    ff = calculate_fat_fraction(water, fat, mask)

    results.append({
        "subject": row["subject"],
        "timepoint": row["timepoint"],
        "slice_id": row["slice_id"],
        "percent_along_muscle": row["percent_along_muscle"],
        "distance_from_muscle_start_mm": row["distance_from_muscle_start_mm"],
        "mean_fat_fraction_percent": np.nanmean(ff),
        "median_fat_fraction_percent": np.nanmedian(ff),
        "std_fat_fraction_percent": np.nanstd(ff),
        "n_muscle_pixels": int(mask.sum()),
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# - How measurements were summarized across locations along the muscle
# The reusable summary file is a CSV file. This is the output we could use in a larger project analysis. The output_file is linked to the TA_fat_fraction_summary.csv and the output file is the results_df.to_csv. 

# Plotting millimeters is important because the slice positions are different between TP1 and TP2. 

# Each of the 8 sampled locations in a visit gets its own mean, median, SD, and pixel count. These are collected in one table. 

# The mean fat fraction against distance from the muscle start (mm) gives a fat-fraction profile along the muscle. 

# If you compare visits or use interpolation in your final analysis, describe those methods clearly, including the variables used and the assumptions made.

# Each figure should have labeled axes, an informative title, and a caption explaining what is shown, why it is included, and what conclusion can be drawn from it.
def summarize_visit(timepoint):
    visit_results = []
    visit_manifest = manifest[manifest["timepoint"] == timepoint]

    for _, row in visit_manifest.iterrows():
        water = np.load(DATA_DIR / row["water_file"])
        fat = np.load(DATA_DIR / row["fat_file"])
        mask = np.load(DATA_DIR / row["mask_file"]).astype(bool)
        ff = calculate_fat_fraction(water, fat, mask)

        visit_results.append({
            "timepoint": timepoint,
            "slice_id": row["slice_id"],
            "percent_along_muscle": row["percent_along_muscle"],
            "distance_from_muscle_start_mm": row["distance_from_muscle_start_mm"],
            "mean_fat_fraction_percent": np.nanmean(ff),
        })

    return pd.DataFrame(visit_results)

profiles_df = pd.concat(
    [summarize_visit("TP1"), summarize_visit("TP2")],
    ignore_index=True
)

plt.figure(figsize=(7, 4))
for timepoint, group in profiles_df.groupby("timepoint"):
    group = group.sort_values("distance_from_muscle_start_mm")
    plt.plot(
        group["distance_from_muscle_start_mm"],   # fixed: was percent_along_muscle
        group["mean_fat_fraction_percent"],
        marker="o",
        label=timepoint
    )
plt.xlabel("Distance from muscle start (mm)")
plt.ylabel("Mean TA fat fraction (%)")
plt.title("Mean fat fraction along the tibialis anterior")
plt.legend(title="Visit")
plt.grid(True, alpha=0.3)
plt.show()

output_file = Path("TA_fat_fraction_summary.csv")
results_df.to_csv(output_file, index=False)
print(f"Saved {output_file}")

## Interpolation Analysis:

Explain how interpolation was used to estimate fat fraction or another measurement at a location between sampled points along the tibialis anterior.

Include:

- The variable used to define location along the muscle, such as distance in millimeters or percentage of muscle length
- The sampled locations used as inputs to the interpolation
- The interpolation method and Python code used
- A plot showing the measured values and the interpolated estimate(s)
- Why interpolation is an estimate rather than a new measurement
- The assumptions made about how the measurement changes between sampled locations
- Whether the estimate is within the range of sampled data or represents extrapolation
- Any limitations or sources of uncertainty in the interpolated result

## Conclusions:

Explain what your analysis suggests about measuring FSHD progression.

Discuss:

- Which MRI, biomarker, or functional measures changed most consistently
- Which measures appeared most sensitive to differences across muscle locations or visits
- Whether the measures could be useful as biomarkers or exploratory clinical-trial endpoints
- What additional evidence would be needed before using them to evaluate treatment effectiveness
- Potential ethical implications of selecting one measure over another, including patient relevance, accessibility, and the risk of overlooking meaningful changes in function

## Limitations and Future Work:

Describe the limitations of your project, including limitations of the sample size, available visits, MRI measurements, segmentation, biomarkers, functional measures, and analysis methods.

If you had more time or additional data, explain how you would improve the analysis. Examples include analyzing more participants, adding more timepoints, improving segmentation, comparing additional muscles, validating the MRI measure against functional outcomes, or testing whether the measure is sensitive to treatment response.

## AI Usage Statement:
*This is where you will provide a short description on whether AI (and which AI) was used for any of the sections above and whether it helped or why it did not help.*
- We used Claude to summarize the dataset.csv

## GitHub Repository Link:
https://github.com/unj4kw/Module-2-.git 

## References:
Tasca et al. (2021). “Longitudinal study of MRI and functional outcome measures in facioscapulohumeral muscular dystrophy.” 
Wong, C.-J., Friedman, S. D., Snider, L., Bennett, S. R., Jones, T. I., Jones, P. L., Shaw, D. W. W., Blemker, S. S., Riem, L., DuCharme, O., Lemmers, R. J. F. L., van der Maarel, S. M., Wang, L. H., Tawil, R., Statland, J. M., & Tapscott, S. J. (2024). 

Regional and bilateral MRI and gene signatures in facioscapulohumeral dystrophy: implications for clinical trial design and mechanisms of disease progression. Human Molecular Genetics, 33(8), 698–708. https://doi.org/10.1093/hmg/ddae007[cite: 20] 

## NOTES FROM YOUR TEAM: 
*This is where our team is taking notes and recording activity.*

## QUESTIONS FOR YOUR TA: 
*These are questions we have for our TA.*